# 07 — Bifactorial HRV analysis

**Aim:** Characterize each HRV metric as a response to all six pairs of demographic/physiological factors, focusing on factor–factor interactions and conditional differences.

- **Original factor categories are never changed in the input data.** All original categories appear in descriptive analyses and distribution figures.
- **Inferential category harmonization:** BP is grouped as *Optimal/Normal*, *High-normal*, *Stage 1–3*; BMI as *Underweight/Normal*, *Overweight*, *Obesity* to avoid empty and extremely small cells. These are **analytical combinations**, not new clinical diagnoses. Tables explicitly identify which scale is used.
- **Primary inferential framework:** HC3-robust two-way factorial linear models with adjusted main effects and an explicitly tested factor interaction. Highly skewed HRV variables are modeled on a log10 scale, while displayed descriptive statistics remain on their original measurement scale.
- **Secondary framework:** continuous-factor interaction models (age, BMI, SBP, DBP) with sex as a categorical factor, and bootstrapped conditional pairwise effect sizes.
- **Sensitivity:** group support, removal of the upper/lower 1% of a response for a separate fit, and scale choice. Nothing is removed or altered from the source dataset.
- All analyses are cross-sectional associations. No causal or clinical-status interpretation is implied.

**Requirements:** pandas, numpy, scipy, statsmodels, seaborn, matplotlib, IPython. No special post hoc package is needed.

## 0. Imports, definitions and input data

In [1]:
from pathlib import Path
from itertools import combinations
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.formula.api import ols
from IPython.display import display

INPUT_FILE = Path("../../outputs/dataset_final.csv")
OUTPUT_DIR = Path("../../eda_results/07_bifactorial_hrv_analysis")
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
GLOBAL_DIR = FIGURE_DIR / "_global"
for folder in (TABLE_DIR, FIGURE_DIR, GLOBAL_DIR):
    folder.mkdir(parents=True, exist_ok=True)

SEED = 20261125
BOOTSTRAP_N = 350
MIN_CELL_N = 5
MIN_CONTRAST_N = 10
MIN_MODEL_CELL_N = 5
FDR_ALPHA = 0.05

HRV = {
    "t2m_pre_mean_rr": ("mean_rr", "Mean RR", "ms"),
    "t2m_pre_mean_hr": ("mean_hr", "Mean HR", "bpm"),
    "t2m_pre_sdnn": ("sdnn", "SDNN", "ms"),
    "t2m_pre_rmssd": ("rmssd", "RMSSD", "ms"),
    "t2m_pre_hf": ("hf", "HF", "ms²"),
    "t2m_pre_lf": ("lf", "LF", "ms²"),
    "t2m_pre_vlf": ("vlf", "VLF", "ms²"),
}
FACTORS = {
    "age_group": ["60-69", "70-79", "80+"],
    "sex_category": ["Female", "Male"],
    "bp_category": ["Optimal", "Normal", "High-normal", "Stage 1", "Stage 2", "Stage 3"],
    "bmi_category": ["Underweight", "Normal", "Overweight", "Obesity"],
}
FACTOR_LABELS = {"age_group": "Age", "sex_category": "Sex", "bp_category": "Blood pressure", "bmi_category": "BMI category"}
ANALYSIS_COL = {"age_group": "age_group", "sex_category": "sex_category", "bp_category": "bp_analysis", "bmi_category": "bmi_analysis"}
ANALYSIS_LEVELS = {
    "age_group": FACTORS["age_group"], "sex_category": FACTORS["sex_category"],
    "bp_category": ["Optimal/Normal", "High-normal", "Stage 1-3"],
    "bmi_category": ["Underweight/Normal", "Overweight", "Obesity"],
}
PALETTES = {
    "age_group": dict(zip(FACTORS["age_group"], sns.color_palette("viridis", 3))),
    "sex_category": dict(zip(FACTORS["sex_category"], sns.color_palette("colorblind", 2))),
    "bp_category": dict(zip(FACTORS["bp_category"], sns.color_palette("crest", 6))),
    "bmi_category": dict(zip(FACTORS["bmi_category"], sns.color_palette("flare", 4))),
}
ANALYSIS_COLORS = {
    "age_group": dict(zip(ANALYSIS_LEVELS["age_group"], sns.color_palette("viridis", 3))),
    "sex_category": dict(zip(ANALYSIS_LEVELS["sex_category"], sns.color_palette("colorblind", 2))),
    "bp_category": dict(zip(ANALYSIS_LEVELS["bp_category"], sns.color_palette("crest", 3))),
    "bmi_category": dict(zip(ANALYSIS_LEVELS["bmi_category"], sns.color_palette("flare", 3))),
}
PAIRS = list(combinations(FACTORS, 2))
PAIR_NAMES = {pair: pair[0].replace("_group", "").replace("_category", "") + "_x_" + pair[1].replace("_group", "").replace("_category", "") for pair in PAIRS}

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.0)
plt.rcParams.update({"figure.facecolor":"white", "savefig.facecolor":"white", "axes.spines.top":False,
                     "axes.spines.right":False, "axes.titleweight":"semibold", "grid.alpha":0.17})

df = pd.read_csv(INPUT_FILE)
required = list(HRV) + list(FACTORS) + ["age", "imc", "bp_systolic", "bp_diastolic"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")
for col in list(HRV) + ["age", "imc", "bp_systolic", "bp_diastolic"]:
    if not pd.api.types.is_numeric_dtype(df[col]):
        raise TypeError(f"Not numeric: {col}")
for f, order in FACTORS.items():
    unexpected = set(df[f].dropna().unique()) - set(order)
    if unexpected:
        raise ValueError(f"Unexpected categories in {f}: {unexpected}")
if df[required].isna().any().any():
    raise ValueError("The final dataset contains missing values in analysis columns.")

# Local analysis-only columns; no source file is changed.
df["bp_analysis"] = df["bp_category"].replace({"Optimal":"Optimal/Normal", "Normal":"Optimal/Normal",
    "High-normal":"High-normal", "Stage 1":"Stage 1-3", "Stage 2":"Stage 1-3", "Stage 3":"Stage 1-3"})
df["bmi_analysis"] = df["bmi_category"].replace({"Underweight":"Underweight/Normal", "Normal":"Underweight/Normal"})
for f in FACTORS:
    df[ANALYSIS_COL[f]] = pd.Categorical(df[ANALYSIS_COL[f]], categories=ANALYSIS_LEVELS[f], ordered=True)

for y, (slug, label, unit) in HRV.items():
    for pair in PAIRS:
        (FIGURE_DIR / slug / PAIR_NAMES[pair]).mkdir(parents=True, exist_ok=True)
    (FIGURE_DIR / slug / "_summary").mkdir(parents=True, exist_ok=True)

# Preselect outcome modeling scale based exclusively on overall distribution.
SCALES = {}
for col in HRV:
    v = df[col]
    SCALES[col] = "log10" if v.min() > 0 and v.quantile(0.99) / max(v.quantile(0.01), 1e-12) >= 30 else "identity"
scale_table = pd.DataFrame([{"hrv_variable": c, "display_name": HRV[c][1], "model_scale": SCALES[c],
    "ratio_q99_q01": df[c].quantile(.99) / max(df[c].quantile(.01), 1e-12)} for c in HRV])
scale_table.to_csv(TABLE_DIR / "hrv_model_scales.csv", index=False)
print(f"Participants: {len(df)} | HRV metrics: {len(HRV)} | factor pairs: {len(PAIRS)}")
print(f"Results folder: {OUTPUT_DIR}")
display(scale_table.round(3))

Participants: 476 | HRV metrics: 7 | factor pairs: 6
Results folder: ../../eda_results/07_bifactorial_hrv_analysis


,hrv_variable,display_name,model_scale,ratio_q99_q01
0,t2m_pre_mean_rr,Mean RR,identity,1.927
1,t2m_pre_mean_hr,Mean HR,identity,1.934
2,t2m_pre_sdnn,SDNN,log10,52.900
3,t2m_pre_rmssd,RMSSD,log10,34.118
4,t2m_pre_hf,HF,log10,1924.167
5,t2m_pre_lf,LF,log10,643.256
6,t2m_pre_vlf,VLF,log10,726.250


## 1. Factorial cell support and analytical category definitions

Descriptive tables always use original factor categories. Inferential category models use the explicitly reported BP/BMI combinations above. The smaller groups remain visible in the descriptive figures; they are **not silently dropped**.

In [2]:
support_rows = []
for factor_a, factor_b in PAIRS:
    pair = PAIR_NAMES[(factor_a, factor_b)]
    for scale in ("original", "analysis"):
        a = factor_a if scale == "original" else ANALYSIS_COL[factor_a]
        b = factor_b if scale == "original" else ANALYSIS_COL[factor_b]
        levels_a = FACTORS[factor_a] if scale == "original" else ANALYSIS_LEVELS[factor_a]
        levels_b = FACTORS[factor_b] if scale == "original" else ANALYSIS_LEVELS[factor_b]
        tab = pd.crosstab(df[a], df[b]).reindex(index=levels_a, columns=levels_b, fill_value=0)
        for la in levels_a:
            for lb in levels_b:
                n = int(tab.loc[la, lb])
                support_rows.append({"pair":pair, "factor_a":factor_a,"factor_b":factor_b,"category_scheme":scale,
                    "level_a":la, "level_b":lb, "n":n, "empty_cell":n==0,
                    "small_cell_lt5":n<MIN_CELL_N, "small_cell_lt10":n<MIN_CONTRAST_N})
support = pd.DataFrame(support_rows)
support.to_csv(TABLE_DIR / "factorial_cell_counts.csv", index=False)

support_summary = (support.groupby(["pair", "category_scheme"], as_index=False, observed=True)
    .agg(cells=("n", "size"), minimum_cell_n=("n", "min"), empty_cells=("empty_cell", "sum"),
         cells_lt5=("small_cell_lt5", "sum"), cells_lt10=("small_cell_lt10", "sum")))
support_summary.to_csv(TABLE_DIR / "factorial_support_summary.csv", index=False)
display(support_summary)

for pair in PAIRS:
    sub = support.loc[(support.pair == PAIR_NAMES[pair]) & (support.category_scheme == "original")]
    matrix = sub.pivot(index="level_a", columns="level_b", values="n")
    fig, ax = plt.subplots(figsize=(max(7, len(matrix.columns)*1.5+3), max(3.8, len(matrix.index)*.9+2)))
    sns.heatmap(matrix, cmap="Blues", annot=True, fmt="d", linewidths=.8, linecolor="white",
                cbar_kws={"label":"Participants"}, ax=ax)
    ax.set(title=f"Cell counts | {FACTOR_LABELS[pair[0]]} × {FACTOR_LABELS[pair[1]]}",
           xlabel=FACTOR_LABELS[pair[1]], ylabel=FACTOR_LABELS[pair[0]])
    fig.tight_layout()
    fig.savefig(GLOBAL_DIR / f"{PAIR_NAMES[pair]}__cell_counts.png", dpi=170, bbox_inches="tight")
    plt.close(fig)
print("Saved 6 original-category cell count heatmaps.")

,pair,category_scheme,cells,minimum_cell_n,empty_cells,cells_lt5,cells_lt10
0,age_x_bmi,analysis,9,6,0,0,1
1,age_x_bmi,original,12,1,0,1,3
2,age_x_bp,analysis,9,9,0,0,1
3,age_x_bp,original,18,0,1,4,8
4,age_x_sex,analysis,6,10,0,0,0
5,age_x_sex,original,6,10,0,0,0
6,bp_x_bmi,analysis,9,27,0,0,0
7,bp_x_bmi,original,24,0,3,5,10
8,sex_x_bmi,analysis,6,39,0,0,0
9,sex_x_bmi,original,8,4,0,1,1


findfont: Failed to find font weight semibold for DejaVu Sans, now using 700.


Saved 6 original-category cell count heatmaps.


## 2. Full descriptive characterization of each HRV metric × factor pair

In [3]:
descriptive_rows = []
for col, (slug, label, unit) in HRV.items():
    for factor_a, factor_b in PAIRS:
        for scale in ("original", "analysis"):
            a = factor_a if scale == "original" else ANALYSIS_COL[factor_a]
            b = factor_b if scale == "original" else ANALYSIS_COL[factor_b]
            order_a = FACTORS[factor_a] if scale == "original" else ANALYSIS_LEVELS[factor_a]
            order_b = FACTORS[factor_b] if scale == "original" else ANALYSIS_LEVELS[factor_b]
            for la in order_a:
                for lb in order_b:
                    x = df.loc[(df[a] == la) & (df[b] == lb), col].dropna().to_numpy(float)
                    n = len(x)
                    descriptive_rows.append({
                        "pair":PAIR_NAMES[(factor_a,factor_b)], "factor_a":factor_a,"factor_b":factor_b,
                        "category_scheme":scale,"level_a":la,"level_b":lb,"hrv_variable":col,
                        "n":n,"mean":np.mean(x) if n else np.nan,"median":np.median(x) if n else np.nan,
                        "std":np.std(x,ddof=1) if n>=2 else np.nan,
                        "q05":np.percentile(x,5) if n else np.nan,"q25":np.percentile(x,25) if n else np.nan,
                        "q75":np.percentile(x,75) if n else np.nan,"q95":np.percentile(x,95) if n else np.nan,
                        "iqr":stats.iqr(x) if n else np.nan,"min":x.min() if n else np.nan,
                        "max":x.max() if n else np.nan,
                        "skew":stats.skew(x,bias=False) if n>=3 and np.std(x)>0 else np.nan,
                        "inferential_caution":n<MIN_CONTRAST_N,
                    })
descriptives = pd.DataFrame(descriptive_rows)
descriptives.to_csv(TABLE_DIR / "factorial_hrv_descriptives.csv", index=False)
print(f"Descriptive records (including empty cells, both schemes): {len(descriptives)}")
display(descriptives.head(12).round(3))

Descriptive records (including empty cells, both schemes): 875


,pair,factor_a,factor_b,category_scheme,level_a,level_b,hrv_variable,n,mean,median,std,q05,q25,q75,q95,iqr,min,max,skew,inferential_caution
0,age_x_sex,age_group,sex_category,original,60-69,Female,t2m_pre_mean_rr,150,845.400,837.0,112.059,672.25,773.50,926.75,1025.05,153.25,589.0,1296.0,0.356,False
1,age_x_sex,age_group,sex_category,original,60-69,Male,t2m_pre_mean_rr,48,836.021,829.0,125.232,664.85,754.00,920.25,1045.75,166.25,577.0,1140.0,0.256,False
2,age_x_sex,age_group,sex_category,original,70-79,Female,t2m_pre_mean_rr,156,868.526,863.5,131.143,650.25,786.25,957.00,1076.75,170.75,468.0,1203.0,-0.175,False
3,age_x_sex,age_group,sex_category,original,70-79,Male,t2m_pre_mean_rr,85,874.906,897.0,133.418,653.20,778.00,961.00,1088.00,183.00,531.0,1187.0,-0.175,False
4,age_x_sex,age_group,sex_category,original,80+,Female,t2m_pre_mean_rr,27,885.741,888.0,114.786,699.80,787.00,989.50,1044.80,202.50,664.0,1078.0,-0.175,False
5,age_x_sex,age_group,sex_category,original,80+,Male,t2m_pre_mean_rr,10,874.100,885.0,121.479,707.10,780.75,969.25,1024.45,188.50,681.0,1051.0,-0.179,False
6,age_x_sex,age_group,sex_category,analysis,60-69,Female,t2m_pre_mean_rr,150,845.400,837.0,112.059,672.25,773.50,926.75,1025.05,153.25,589.0,1296.0,0.356,False
7,age_x_sex,age_group,sex_category,analysis,60-69,Male,t2m_pre_mean_rr,48,836.021,829.0,125.232,664.85,754.00,920.25,1045.75,166.25,577.0,1140.0,0.256,False
8,age_x_sex,age_group,sex_category,analysis,70-79,Female,t2m_pre_mean_rr,156,868.526,863.5,131.143,650.25,786.25,957.00,1076.75,170.75,468.0,1203.0,-0.175,False
9,age_x_sex,age_group,sex_category,analysis,70-79,Male,t2m_pre_mean_rr,85,874.906,897.0,133.418,653.20,778.00,961.00,1088.00,183.00,531.0,1187.0,-0.175,False


## 3. Two-way factorial models, interaction tests and adjusted main effects

HC3-robust Wald tests of the interaction use a complete two-factor model. Adjusted main associations are tested in an additive model. Factorial partial R² is computed from nested residual sums of squares (descriptive model-effect magnitude, not a robust test statistic). FDR is controlled **separately** for 42 interactions and 84 adjusted main-association tests. Saturated sparse original-category models are **not** reported as reliable inferences: primary tests use the documented analytical grouping.

In [4]:
def prepare_model_frame(col):
    d = df.copy()
    d["y_model"] = np.log10(d[col]) if SCALES[col] == "log10" else d[col].astype(float)
    return d

def get_wald_p(model, term):
    try:
        table = model.wald_test_terms(scalar=True).table
        return float(table.loc[term, "pvalue"])
    except (KeyError, ValueError, np.linalg.LinAlgError):
        return np.nan

def fit_cat_pair(col, factor_a, factor_b, data=None, outcome_scale=None):
    data = df if data is None else data
    da = ANALYSIS_COL[factor_a]
    db = ANALYSIS_COL[factor_b]
    d = data[[da, db, col]].dropna().copy()
    d["y_model"] = np.log10(d[col]) if (outcome_scale or SCALES[col]) == "log10" else d[col].astype(float)
    cell = pd.crosstab(d[da], d[db]).reindex(index=ANALYSIS_LEVELS[factor_a],
        columns=ANALYSIS_LEVELS[factor_b], fill_value=0)
    fit_ok = int(cell.min().min()) >= MIN_MODEL_CELL_N
    if not fit_ok:
        return None, None, None, cell, "insufficient_cell_support"
    fa, fb = f"C({da}, Sum)", f"C({db}, Sum)"
    m0 = ols("y_model ~ 1", data=d).fit()
    ma = ols(f"y_model ~ {fa}", data=d).fit()
    mb = ols(f"y_model ~ {fb}", data=d).fit()
    additive = ols(f"y_model ~ {fa} + {fb}", data=d).fit(cov_type="HC3")
    full = ols(f"y_model ~ {fa} * {fb}", data=d).fit(cov_type="HC3")
    # Guard against singular or nearly singular design matrices.
    if full.model.rank < full.model.exog.shape[1]:
        return None, None, None, cell, "rank_deficient"
    rss0 = float(np.sum(m0.resid**2))
    rssa = float(np.sum(ma.resid**2))
    rssb = float(np.sum(mb.resid**2))
    rss_add = float(np.sum(additive.resid**2))
    rss_full = float(np.sum(full.resid**2))
    magnitude = {
        "partial_r2_a_given_b": max(0.,(rssb-rss_add)/rssb) if rssb>0 else np.nan,
        "partial_r2_b_given_a": max(0.,(rssa-rss_add)/rssa) if rssa>0 else np.nan,
        "interaction_partial_r2": max(0.,(rss_add-rss_full)/rss_add) if rss_add>0 else np.nan,
        "full_model_r2": full.rsquared,
    }
    return additive, full, magnitude, cell, "ok"

model_rows = []
model_store = {}
for col in HRV:
    for a,b in PAIRS:
        pair = PAIR_NAMES[(a,b)]
        add, full, magnitude, cells, status = fit_cat_pair(col,a,b)
        result = {"pair":pair,"factor_a":a,"factor_b":b,"hrv_variable":col,
            "model_scale":SCALES[col],"n":len(df),"minimum_model_cell_n":int(cells.min().min()),
            "model_status":status,"categorical_scheme":"analysis_collapsed_BP_BMI"}
        if status == "ok":
            term_a, term_b = f"C({ANALYSIS_COL[a]}, Sum)", f"C({ANALYSIS_COL[b]}, Sum)"
            result.update({"main_a_p_hc3":get_wald_p(add,term_a),"main_b_p_hc3":get_wald_p(add,term_b),
                "interaction_p_hc3":get_wald_p(full, f"{term_a}:{term_b}"), **magnitude})
            model_store[(col,pair)] = (add,full)
        model_rows.append(result)
cat_models = pd.DataFrame(model_rows)
for pc,qc in [("interaction_p_hc3","interaction_q_bh_42"),
              ("main_a_p_hc3","main_a_q_bh_84"),("main_b_p_hc3","main_b_q_bh_84")]:
    # Both adjusted-main-effect columns are corrected as one family of 84 tests.
    if pc == "main_a_p_hc3":
        continue
    if pc == "main_b_p_hc3":
        p_all = np.concatenate([cat_models["main_a_p_hc3"].to_numpy(float),cat_models["main_b_p_hc3"].to_numpy(float)])
        q_all = np.full(len(p_all),np.nan)
        valid = np.isfinite(p_all)
        q_all[valid] = multipletests(p_all[valid],method="fdr_bh")[1]
        cat_models["main_a_q_bh_84"] = q_all[:len(cat_models)]
        cat_models["main_b_q_bh_84"] = q_all[len(cat_models):]
    else:
        pvals = cat_models[pc].to_numpy(float)
        cat_models[qc] = np.nan
        valid = np.isfinite(pvals)
        cat_models.loc[valid,qc] = multipletests(pvals[valid],method="fdr_bh")[1]
cat_models.to_csv(TABLE_DIR / "categorical_factorial_models.csv",index=False)
print(f"Categorical model comparisons: {len(cat_models)} | estimated: {(cat_models.model_status=='ok').sum()}")
display(cat_models.round(4))

Categorical model comparisons: 42 | estimated: 42


,pair,factor_a,factor_b,hrv_variable,model_scale,n,minimum_model_cell_n,model_status,categorical_scheme,main_a_p_hc3,main_b_p_hc3,interaction_p_hc3,partial_r2_a_given_b,partial_r2_b_given_a,interaction_partial_r2,full_model_r2,interaction_q_bh_42,main_a_q_bh_84,main_b_q_bh_84
0,age_x_sex,age_group,sex_category,t2m_pre_mean_rr,identity,476,10,ok,analysis_collapsed_BP_BMI,0.0292,0.9530,0.8234,0.0140,0.0000,0.0009,0.0149,0.9285,0.1219,0.9645
1,age_x_bp,age_group,bp_category,t2m_pre_mean_rr,identity,476,9,ok,analysis_collapsed_BP_BMI,0.0279,0.2716,0.0022,0.0140,0.0058,0.0233,0.0426,0.0366,0.1219,0.5045
2,age_x_bmi,age_group,bmi_category,t2m_pre_mean_rr,identity,476,6,ok,analysis_collapsed_BP_BMI,0.0456,0.4590,0.3692,0.0123,0.0036,0.0085,0.0259,0.8048,0.1667,0.6649
3,sex_x_bp,sex_category,bp_category,t2m_pre_mean_rr,identity,476,35,ok,analysis_collapsed_BP_BMI,0.7238,0.2519,0.2295,0.0003,0.0060,0.0069,0.0129,0.8048,0.8427,0.5045
4,sex_x_bmi,sex_category,bmi_category,t2m_pre_mean_rr,identity,476,39,ok,analysis_collapsed_BP_BMI,0.8291,0.3025,0.8646,0.0001,0.0053,0.0007,0.0061,0.9285,0.8770,0.5045
5,bp_x_bmi,bp_category,bmi_category,t2m_pre_mean_rr,identity,476,27,ok,analysis_collapsed_BP_BMI,0.2862,0.3308,0.7037,0.0054,0.0049,0.0041,0.0148,0.9237,0.5045,0.5076
6,age_x_sex,age_group,sex_category,t2m_pre_mean_hr,identity,476,10,ok,analysis_collapsed_BP_BMI,0.0630,0.8787,0.7977,0.0101,0.0001,0.0010,0.0111,0.9285,0.2117,0.9113
7,age_x_bp,age_group,bp_category,t2m_pre_mean_hr,identity,476,9,ok,analysis_collapsed_BP_BMI,0.0585,0.2371,0.0026,0.0102,0.0066,0.0199,0.0362,0.0366,0.2048,0.5045
8,age_x_bmi,age_group,bmi_category,t2m_pre_mean_hr,identity,476,6,ok,analysis_collapsed_BP_BMI,0.1079,0.4591,0.2540,0.0083,0.0036,0.0103,0.0237,0.8048,0.3356,0.6649
9,sex_x_bp,sex_category,bp_category,t2m_pre_mean_hr,identity,476,35,ok,analysis_collapsed_BP_BMI,0.8352,0.2285,0.2238,0.0001,0.0066,0.0072,0.0137,0.8048,0.8770,0.5045


## 4. Conditional group contrasts, effect sizes, uncertainty and FDR

Each test compares levels of factor A **within** one level of factor B, and levels of B **within** each level of A. Analysis-category cells with n < 10 are omitted from inferential contrasts and flagged in the cell-support table. Tests use two-sided Mann–Whitney U, Cliff's delta and the median difference with percentile bootstrap 95% CIs, Kolmogorov–Smirnov, Wasserstein distance and Brown–Forsythe dispersion tests. BH correction is applied separately per HRV metric × factor pair across all performed simple comparisons; a separate global BH q-value is also supplied across all comparisons. These are **exploratory contrasts**, not evidence for an interaction on their own.

In [5]:
def bootstrap_effects(x, y, seed, repeats=BOOTSTRAP_N):
    rng = np.random.default_rng(seed)
    cliffs = np.empty(repeats)
    medians = np.empty(repeats)
    for i in range(repeats):
        bx = x[rng.integers(0,len(x),len(x))]
        by = y[rng.integers(0,len(y),len(y))]
        u = stats.mannwhitneyu(bx,by,alternative="two-sided",method="asymptotic").statistic
        cliffs[i] = 2*u/(len(bx)*len(by))-1
        medians[i] = np.median(bx)-np.median(by)
    return [float(z) for z in np.quantile(cliffs,[.025,.975])], [float(z) for z in np.quantile(medians,[.025,.975])]

contrast_rows = []
seed_idx = 0
for col in HRV:
    for a,b in PAIRS:
        pair = PAIR_NAMES[(a,b)]
        for focal, conditioning in ((a,b),(b,a)):
            focal_col, cond_col = ANALYSIS_COL[focal],ANALYSIS_COL[conditioning]
            for cond_level in ANALYSIS_LEVELS[conditioning]:
                ds = df.loc[df[cond_col]==cond_level]
                for lv1,lv2 in combinations(ANALYSIS_LEVELS[focal],2):
                    x = ds.loc[ds[focal_col]==lv1,col].dropna().to_numpy(float)
                    y = ds.loc[ds[focal_col]==lv2,col].dropna().to_numpy(float)
                    if min(len(x),len(y)) < MIN_CONTRAST_N:
                        continue
                    u_result = stats.mannwhitneyu(x,y,alternative="two-sided",method="asymptotic")
                    delta = 2*u_result.statistic/(len(x)*len(y))-1
                    deltaci,medci = bootstrap_effects(x,y,SEED+100_000+seed_idx)
                    ks = stats.ks_2samp(x,y,alternative="two-sided",method="auto")
                    bf = stats.levene(x,y,center="median")
                    wasserstein = stats.wasserstein_distance(x,y)
                    seed_idx+=1
                    contrast_rows.append({"pair":pair,"hrv_variable":col,"focal_factor":focal,"conditioning_factor":conditioning,
                        "conditioning_level":cond_level,"level_1":lv1,"level_2":lv2,"n_1":len(x),"n_2":len(y),
                        "median_1":np.median(x),"median_2":np.median(y),"median_diff_1_minus_2":np.median(x)-np.median(y),
                        "median_diff_ci95_low":medci[0],"median_diff_ci95_high":medci[1],
                        "cliffs_delta_1_vs_2":delta,"cliffs_ci95_low":deltaci[0],"cliffs_ci95_high":deltaci[1],
                        "mw_u":u_result.statistic,"p_two_sided":u_result.pvalue,
                        "ks_statistic":ks.statistic,"ks_p_two_sided":ks.pvalue,
                        "wasserstein_distance":wasserstein,"brown_forsythe_statistic":bf.statistic,
                        "brown_forsythe_p":bf.pvalue})
conditional = pd.DataFrame(contrast_rows)
conditional["q_bh_within_hrv_pair"] = np.nan
for (_, _), idx in conditional.groupby(["hrv_variable","pair"]).groups.items():
    ids = list(idx)
    conditional.loc[ids,"q_bh_within_hrv_pair"] = multipletests(conditional.loc[ids,"p_two_sided"],method="fdr_bh")[1]
conditional["q_bh_global"] = multipletests(conditional["p_two_sided"],method="fdr_bh")[1]
for original_p, local_q, global_q in [
    ("ks_p_two_sided","ks_q_bh_within_hrv_pair","ks_q_bh_global"),
    ("brown_forsythe_p","brown_forsythe_q_bh_within_hrv_pair","brown_forsythe_q_bh_global"),
]:
    conditional[local_q] = np.nan
    for (_, _), idx in conditional.groupby(["hrv_variable","pair"]).groups.items():
        ids = list(idx)
        conditional.loc[ids,local_q] = multipletests(conditional.loc[ids,original_p],method="fdr_bh")[1]
    conditional[global_q] = multipletests(conditional[original_p],method="fdr_bh")[1]
conditional.to_csv(TABLE_DIR / "conditional_pairwise_effects.csv",index=False)
print(f"Conditional comparisons (eligible n≥{MIN_CONTRAST_N}): {len(conditional)}")
display(conditional.head(10).round(3))

Conditional comparisons (eligible n≥10): 511


,pair,hrv_variable,focal_factor,conditioning_factor,conditioning_level,level_1,level_2,n_1,n_2,median_1,...,ks_p_two_sided,wasserstein_distance,brown_forsythe_statistic,brown_forsythe_p,q_bh_within_hrv_pair,q_bh_global,ks_q_bh_within_hrv_pair,ks_q_bh_global,brown_forsythe_q_bh_within_hrv_pair,brown_forsythe_q_bh_global
0,age_x_sex,t2m_pre_mean_rr,age_group,sex_category,Female,60-69,70-79,150,156,837.0,...,0.130,30.759,2.499,0.115,0.248,0.445,0.391,0.707,0.969,0.761
1,age_x_sex,t2m_pre_mean_rr,age_group,sex_category,Female,60-69,80+,150,27,837.0,...,0.057,43.256,0.010,0.921,0.248,0.474,0.323,0.563,0.969,0.969
2,age_x_sex,t2m_pre_mean_rr,age_group,sex_category,Female,70-79,80+,156,27,863.5,...,0.516,25.563,0.539,0.464,0.832,0.842,0.774,0.912,0.969,0.817
3,age_x_sex,t2m_pre_mean_rr,age_group,sex_category,Male,60-69,70-79,48,85,829.0,...,0.072,41.178,0.187,0.666,0.248,0.495,0.323,0.573,0.969,0.909
4,age_x_sex,t2m_pre_mean_rr,age_group,sex_category,Male,60-69,80+,48,10,829.0,...,0.675,43.696,0.002,0.969,0.832,0.752,0.868,0.922,0.969,0.984
5,age_x_sex,t2m_pre_mean_rr,age_group,sex_category,Male,70-79,80+,85,10,897.0,...,0.998,22.629,0.071,0.791,0.985,1.000,0.998,1.000,0.969,0.949
6,age_x_sex,t2m_pre_mean_rr,sex_category,age_group,60-69,Female,Male,150,48,837.0,...,0.437,18.103,1.159,0.283,0.832,0.906,0.774,0.890,0.969,0.761
7,age_x_sex,t2m_pre_mean_rr,sex_category,age_group,70-79,Female,Male,156,85,863.5,...,0.299,13.976,0.198,0.657,0.863,0.938,0.672,0.800,0.969,0.904
8,age_x_sex,t2m_pre_mean_rr,sex_category,age_group,80+,Female,Male,27,10,888.0,...,0.998,21.952,0.164,0.688,0.897,0.946,0.998,1.000,0.969,0.915
9,age_x_bp,t2m_pre_mean_rr,age_group,bp_category,Optimal/Normal,60-69,70-79,90,115,850.5,...,0.398,31.162,2.789,0.096,0.566,0.679,0.646,0.870,0.376,0.761


## 5. Continuous and mixed-factor interaction models

Nine planned specifications cover all six factor pairs: age×sex, age×BMI, age×SBP/DBP, sex×BMI, sex×SBP/DBP, BMI×SBP/DBP. Continuous predictors are centered and rescaled for interpretable coefficients (age /10 years, BMI /5 kg/m², BP /10 mmHg). The interaction is tested in an HC3-robust model; additive associations are assessed separately. FDR for the **63 continuous/mixed interactions** is computed as one family. This supplements, but does not replace, the categorical factorial analysis.

In [6]:
CONTINUOUS_DEFS = {
    "age":("age_10","Age",10.0), "imc":("bmi_5","BMI",5.0),
    "bp_systolic":("sbp_10","SBP",10.0),"bp_diastolic":("dbp_10","DBP",10.0),
}
for raw,(scaled,label,step) in CONTINUOUS_DEFS.items():
    df[scaled]=(df[raw]-df[raw].mean())/step

SPECS = [
    ("age_x_sex","age", "sex_category", "age_group","sex_category"),
    ("age_x_bmi","age", "imc", "age_group","bmi_category"),
    ("age_x_bp_sbp","age", "bp_systolic", "age_group","bp_category"),
    ("age_x_bp_dbp","age", "bp_diastolic", "age_group","bp_category"),
    ("sex_x_bmi","imc", "sex_category", "bmi_category","sex_category"),
    ("sex_x_bp_sbp","bp_systolic", "sex_category", "bp_category","sex_category"),
    ("sex_x_bp_dbp","bp_diastolic", "sex_category", "bp_category","sex_category"),
    ("bmi_x_bp_sbp","imc", "bp_systolic", "bmi_category","bp_category"),
    ("bmi_x_bp_dbp","imc", "bp_diastolic", "bmi_category","bp_category"),
]
continuous_rows=[]
continuous_models={}
for col in HRV:
    for spec_name, x_raw, z_raw, color_x, color_z in SPECS:
        xx=CONTINUOUS_DEFS[x_raw][0]
        zz= f"C({z_raw}, Sum)" if z_raw=="sex_category" else CONTINUOUS_DEFS[z_raw][0]
        d=df.copy()
        d["y_model"] = np.log10(d[col]) if SCALES[col]=="log10" else d[col]
        add=ols(f"y_model ~ {xx} + {zz}",data=d).fit(cov_type="HC3")
        full=ols(f"y_model ~ {xx} * {zz}",data=d).fit(cov_type="HC3")
        rss_add=float(np.sum(add.resid**2));rss_full=float(np.sum(full.resid**2))
        term=f"{xx}:{zz}"
        p=get_wald_p(full,term)
        continuous_rows.append({"specification":spec_name,"hrv_variable":col,"x_variable":x_raw,"z_variable":z_raw,
            "x_rescaled":xx,"z_model_term":zz,"x_unit_step":CONTINUOUS_DEFS[x_raw][2],
            "z_unit_step":CONTINUOUS_DEFS[z_raw][2] if z_raw!="sex_category" else np.nan,
            "outcome_scale":SCALES[col],"n":len(d),"interaction_p_hc3":p,
            "interaction_partial_r2":max(0.,(rss_add-rss_full)/rss_add) if rss_add>0 else np.nan,
            "additive_main_x_p_hc3":get_wald_p(add,xx),"additive_main_z_p_hc3":get_wald_p(add,zz),
            "interaction_parameter":full.params.get(term,np.nan),
            "interaction_parameter_ci95_low":full.conf_int().loc[term,0] if term in full.params.index else np.nan,
            "interaction_parameter_ci95_high":full.conf_int().loc[term,1] if term in full.params.index else np.nan,
            "full_model_r2":full.rsquared})
        continuous_models[(col,spec_name)] = full
continuous_results = pd.DataFrame(continuous_rows)
continuous_results["interaction_q_bh_63"] = multipletests(continuous_results["interaction_p_hc3"],method="fdr_bh")[1]
for pcol,qcol in [("additive_main_x_p_hc3","main_x_q_bh_126"),("additive_main_z_p_hc3","main_z_q_bh_126")]:
    pass
both = np.r_[continuous_results["additive_main_x_p_hc3"].values,continuous_results["additive_main_z_p_hc3"].values]
qboth=multipletests(both,method="fdr_bh")[1]
continuous_results["main_x_q_bh_126"] = qboth[:len(continuous_results)]
continuous_results["main_z_q_bh_126"] = qboth[len(continuous_results):]
continuous_results.to_csv(TABLE_DIR / "continuous_mixed_interaction_models.csv",index=False)
print(f"Continuous/mixed interaction analyses: {len(continuous_results)}")
display(continuous_results.head(12).round(4))

Continuous/mixed interaction analyses: 63


,specification,hrv_variable,x_variable,z_variable,x_rescaled,z_model_term,x_unit_step,z_unit_step,outcome_scale,n,...,interaction_partial_r2,additive_main_x_p_hc3,additive_main_z_p_hc3,interaction_parameter,interaction_parameter_ci95_low,interaction_parameter_ci95_high,full_model_r2,interaction_q_bh_63,main_x_q_bh_126,main_z_q_bh_126
0,age_x_sex,t2m_pre_mean_rr,age,sex_category,age_10,"C(sex_category, Sum)",10.0,NaN,identity,476,...,0.0001,0.0037,0.9546,NaN,NaN,NaN,0.0164,0.9857,0.0265,0.9700
1,age_x_bmi,t2m_pre_mean_rr,age,imc,age_10,bmi_5,10.0,5.0,identity,476,...,0.0003,0.0047,0.5478,-3.4381,-22.3018,15.4256,0.0172,0.9857,0.0265,0.7389
2,age_x_bp_sbp,t2m_pre_mean_rr,age,bp_systolic,age_10,sbp_10,10.0,10.0,identity,476,...,0.0000,0.0038,0.3129,0.5673,-10.1889,11.3235,0.0187,0.9857,0.0265,0.5928
3,age_x_bp_dbp,t2m_pre_mean_rr,age,bp_diastolic,age_10,dbp_10,10.0,10.0,identity,476,...,0.0006,0.0526,0.0000,4.6376,-16.0166,25.2919,0.0588,0.9857,0.1948,0.0002
4,sex_x_bmi,t2m_pre_mean_rr,imc,sex_category,bmi_5,"C(sex_category, Sum)",5.0,NaN,identity,476,...,0.0000,0.3524,0.8805,NaN,NaN,NaN,0.0018,0.9857,0.6355,0.9402
5,sex_x_bp_sbp,t2m_pre_mean_rr,bp_systolic,sex_category,sbp_10,"C(sex_category, Sum)",10.0,NaN,identity,476,...,0.0004,0.3029,0.8977,NaN,NaN,NaN,0.0030,0.9857,0.5894,0.9415
6,sex_x_bp_dbp,t2m_pre_mean_rr,bp_diastolic,sex_category,dbp_10,"C(sex_category, Sum)",10.0,NaN,identity,476,...,0.0000,0.0000,0.4672,NaN,NaN,NaN,0.0527,0.9857,0.0000,0.7092
7,bmi_x_bp_sbp,t2m_pre_mean_rr,imc,bp_systolic,bmi_5,sbp_10,5.0,10.0,identity,476,...,0.0141,0.3041,0.2639,-8.2720,-15.3379,-1.2062,0.0187,0.2779,0.5894,0.5611
8,bmi_x_bp_dbp,t2m_pre_mean_rr,imc,bp_diastolic,bmi_5,dbp_10,5.0,10.0,identity,476,...,0.0011,0.6002,0.0000,-3.4545,-14.2079,7.2990,0.0531,0.9857,0.7563,0.0001
9,age_x_sex,t2m_pre_mean_hr,age,sex_category,age_10,"C(sex_category, Sum)",10.0,NaN,identity,476,...,0.0001,0.0046,0.9292,NaN,NaN,NaN,0.0145,0.9857,0.0265,0.9597


## 6. Sensitivity analyses

For each categorical pair, refit the HC3 interaction after removing the upper and lower 1% **of that response only**, separately from the original dataset. On the same full sample, also compare log10 versus untransformed-response interaction p-values. These are robustness checks and do not change any participant's status in the analysis dataset.

In [7]:
sensitivity_rows=[]
for col in HRV:
    raw_scale=SCALES[col]
    alternative="identity" if raw_scale=="log10" else "log10"
    cutoff_lo,cutoff_hi=df[col].quantile([.01,.99])
    trimmed=df.loc[df[col].between(cutoff_lo,cutoff_hi)].copy()
    for a,b in PAIRS:
        pair=PAIR_NAMES[(a,b)]
        primary=cat_models.loc[(cat_models.hrv_variable==col)&(cat_models.pair==pair)].iloc[0]
        for sensitivity_type,d,scale in [
            ("exclude_outer_1pct_per_tail",trimmed,raw_scale),
            ("alternative_outcome_scale",df,alternative),
        ]:
            add,full,mag,cells,status=fit_cat_pair(col,a,b,data=d,outcome_scale=scale)
            if status=="ok":
                ia=f"C({ANALYSIS_COL[a]}, Sum):C({ANALYSIS_COL[b]}, Sum)"
                pp=get_wald_p(full,ia)
                pr2=mag["interaction_partial_r2"]
            else:
                pp,pr2=np.nan,np.nan
            sensitivity_rows.append({"hrv_variable":col,"pair":pair,"sensitivity":sensitivity_type,
                "primary_scale":raw_scale,"sensitivity_scale":scale,"primary_n":len(df),"sensitivity_n":len(d),
                "minimum_cell_n":int(cells.min().min()),"status":status,
                "primary_interaction_p_hc3":primary.interaction_p_hc3,
                "primary_interaction_partial_r2":primary.interaction_partial_r2,
                "sensitivity_interaction_p_hc3":pp,"sensitivity_interaction_partial_r2":pr2,
                "delta_partial_r2":pr2-primary.interaction_partial_r2 if np.isfinite(pr2) else np.nan,
                "trim_q01":cutoff_lo if sensitivity_type.startswith("exclude") else np.nan,
                "trim_q99":cutoff_hi if sensitivity_type.startswith("exclude") else np.nan})
sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity.to_csv(TABLE_DIR / "categorical_interaction_sensitivity.csv",index=False)
print(f"Sensitivity refits: {len(sensitivity)}")
display(sensitivity.head(12).round(4))

Sensitivity refits: 84


,hrv_variable,pair,sensitivity,primary_scale,sensitivity_scale,primary_n,sensitivity_n,minimum_cell_n,status,primary_interaction_p_hc3,primary_interaction_partial_r2,sensitivity_interaction_p_hc3,sensitivity_interaction_partial_r2,delta_partial_r2,trim_q01,trim_q99
0,t2m_pre_mean_rr,age_x_sex,exclude_outer_1pct_per_tail,identity,identity,476,466,10,ok,0.8234,0.0009,0.8899,0.0005,-0.0003,586.0,1129.25
1,t2m_pre_mean_rr,age_x_sex,alternative_outcome_scale,identity,log10,476,476,10,ok,0.8234,0.0009,0.7989,0.0010,0.0001,NaN,NaN
2,t2m_pre_mean_rr,age_x_bp,exclude_outer_1pct_per_tail,identity,identity,476,466,9,ok,0.0022,0.0233,0.0033,0.0256,0.0023,586.0,1129.25
3,t2m_pre_mean_rr,age_x_bp,alternative_outcome_scale,identity,log10,476,476,9,ok,0.0022,0.0233,0.0026,0.0217,-0.0016,NaN,NaN
4,t2m_pre_mean_rr,age_x_bmi,exclude_outer_1pct_per_tail,identity,identity,476,466,6,ok,0.3692,0.0085,0.3966,0.0091,0.0005,586.0,1129.25
5,t2m_pre_mean_rr,age_x_bmi,alternative_outcome_scale,identity,log10,476,476,6,ok,0.3692,0.0085,0.3292,0.0091,0.0006,NaN,NaN
6,t2m_pre_mean_rr,sex_x_bp,exclude_outer_1pct_per_tail,identity,identity,476,466,35,ok,0.2295,0.0069,0.3124,0.0055,-0.0014,586.0,1129.25
7,t2m_pre_mean_rr,sex_x_bp,alternative_outcome_scale,identity,log10,476,476,35,ok,0.2295,0.0069,0.2220,0.0071,0.0002,NaN,NaN
8,t2m_pre_mean_rr,sex_x_bmi,exclude_outer_1pct_per_tail,identity,identity,476,466,38,ok,0.8646,0.0007,0.9639,0.0002,-0.0006,586.0,1129.25
9,t2m_pre_mean_rr,sex_x_bmi,alternative_outcome_scale,identity,log10,476,476,39,ok,0.8646,0.0007,0.9381,0.0003,-0.0004,NaN,NaN


## 7. Figures — grouped by HRV metric and factor pair

Each HRV metric has six factor-pair subfolders. Visuals use the **original** factor levels for observed distributions and the **analysis-category** levels for interaction-model estimates and conditional effect comparisons. Colors correspond to the factor displayed in each plot; legends explicitly identify it.

In [8]:
def use_log_axis(col):
    v=df[col].to_numpy(float)
    return bool(v.min()>0 and np.quantile(v,.99)/np.quantile(v,.01)>30)

def save_figure(fig, path, dpi=155):
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)

def matrix_for(col,a,b,scheme,value):
    d=descriptives.loc[(descriptives.hrv_variable==col)&(descriptives.pair==PAIR_NAMES[(a,b)])&
                       (descriptives.category_scheme==scheme)]
    la=FACTORS[a] if scheme=="original" else ANALYSIS_LEVELS[a]
    lb=FACTORS[b] if scheme=="original" else ANALYSIS_LEVELS[b]
    return d.pivot(index="level_a",columns="level_b",values=value).reindex(index=la,columns=lb)

# Original-category distribution views: grouped boxplots, cell medians/IQRs and count heatmaps.
for col,(slug,label,unit) in HRV.items():
    for a,b in PAIRS:
        folder=FIGURE_DIR/slug/PAIR_NAMES[(a,b)]
        log_y=use_log_axis(col)
        fig, axs=plt.subplots(2,2,figsize=(18.8,12.2))
        sns.boxplot(data=df,x=a,y=col,hue=b,order=FACTORS[a],hue_order=FACTORS[b],
                    palette=PALETTES[b],showfliers=False,ax=axs[0,0])
        if log_y: axs[0,0].set_yscale("log")
        axs[0,0].set(title="Observed group distributions (boxplots)",xlabel=FACTOR_LABELS[a],ylabel=f"{label} ({unit})")
        axs[0,0].legend(title=FACTOR_LABELS[b],fontsize=8,loc="best",ncol=2)
        axs[0,0].tick_params(axis="x",rotation=20)
        mat=matrix_for(col,a,b,"original","median")
        ns=matrix_for(col,a,b,"original","n")
        annot=np.empty(mat.shape,dtype=object)
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                annot[i,j]=f"{mat.iloc[i,j]:.2g}\nn={int(ns.iloc[i,j])}" if ns.iloc[i,j]>0 else "—"
        sns.heatmap(mat,ax=axs[0,1],cmap="YlGnBu",annot=annot,fmt="",linewidths=.4,mask=ns.eq(0),
                    cbar_kws={"label":f"Median {label} ({unit})"},annot_kws={"size":8})
        axs[0,1].set(title="Cell medians and sample sizes",xlabel=FACTOR_LABELS[b],ylabel=FACTOR_LABELS[a])
        sns.heatmap(matrix_for(col,a,b,"original","iqr"),ax=axs[1,0],cmap="mako",annot=True,fmt=".1f",
                    linewidths=.4,cbar_kws={"label":f"IQR ({unit})"},annot_kws={"size":8})
        axs[1,0].set(title="Within-cell dispersion (IQR)",xlabel=FACTOR_LABELS[b],ylabel=FACTOR_LABELS[a])
        sns.heatmap(ns,ax=axs[1,1],cmap="Blues",annot=True,fmt=".0f",linewidths=.4,
                    cbar_kws={"label":"Participants"})
        axs[1,1].set(title="Cell support (original categories)",xlabel=FACTOR_LABELS[b],ylabel=FACTOR_LABELS[a])
        fig.suptitle(f"{label} | {FACTOR_LABELS[a]} × {FACTOR_LABELS[b]} — descriptive overview",
                     fontsize=19,y=.998,fontweight="semibold")
        fig.tight_layout(rect=(0,0,1,.97),h_pad=2.3,w_pad=1.5)
        save_figure(fig,folder/"01_descriptive_overview.png")

        # Conditioned ECDF views of factor A, separately within every original B level.
        nlevels=len(FACTORS[b]); ncols=min(3,nlevels); nrows=int(np.ceil(nlevels/ncols))
        fig, axes=plt.subplots(nrows,ncols,figsize=(6.0*ncols,4.4*nrows),squeeze=False)
        for ax,lvb in zip(axes.flat,FACTORS[b]):
            sub=df.loc[df[b]==lvb]
            if len(sub):
                sns.ecdfplot(data=sub,x=col,hue=a,hue_order=FACTORS[a],palette=PALETTES[a],
                             linewidth=1.8,ax=ax)
            ax.set(title=f"{FACTOR_LABELS[b]} = {lvb} | n={len(sub)}",xlabel=f"{label} ({unit})",ylabel="Cumulative fraction")
            if log_y: ax.set_xscale("log")
            if ax.legend_: ax.legend_.set_title(FACTOR_LABELS[a])
        for ax in list(axes.flat)[nlevels:]: ax.axis("off")
        fig.suptitle(f"{label}: conditioned ECDFs by {FACTOR_LABELS[b]}",fontsize=18,fontweight="semibold",y=.997)
        fig.tight_layout(rect=(0,0,1,.95),h_pad=2,w_pad=1.4)
        save_figure(fig,folder/"02_conditioned_ecdf.png")
print("Saved descriptive figures: 2 per HRV metric × factor pair (84 PNG).")

findfont: Failed to find font weight semibold for DejaVu Sans, now using 700.
findfont: Failed to find font weight semibold for DejaVu Sans, now using 700.


Saved descriptive figures: 2 per HRV metric × factor pair (84 PNG).


## 8. Figures — interaction estimates and conditioned effect contrasts

Model plots show cell-level **arithmetic means** if modeled on the original outcome scale, or back-transformed **geometric means** if log10 modeled. These are not medians. Error bars are HC3-robust 95% confidence intervals. Forest plots show signed Cliff's delta (level 1 versus level 2) within the stated conditioning factor level.

In [9]:
for col,(slug,label,unit) in HRV.items():
    for a,b in PAIRS:
        pair=PAIR_NAMES[(a,b)]
        folder=FIGURE_DIR/slug/pair
        model_row=cat_models.loc[(cat_models.hrv_variable==col)&(cat_models.pair==pair)].iloc[0]
        if model_row.model_status!="ok":
            continue
        fit=model_store[(col,pair)][1]
        aa=ANALYSIS_COL[a]; bb=ANALYSIS_COL[b]
        grid=pd.DataFrame([{aa:la,bb:lb} for la in ANALYSIS_LEVELS[a] for lb in ANALYSIS_LEVELS[b]])
        grid[aa]=pd.Categorical(grid[aa],categories=ANALYSIS_LEVELS[a],ordered=True)
        grid[bb]=pd.Categorical(grid[bb],categories=ANALYSIS_LEVELS[b],ordered=True)
        pred=fit.get_prediction(grid).summary_frame(alpha=.05)
        for name,field in [("estimate","mean"),("ci_low","mean_ci_lower"),("ci_high","mean_ci_upper")]:
            grid[name]=10**pred[field].to_numpy() if SCALES[col]=="log10" else pred[field].to_numpy()
        estimates_label="HC3 geometric mean" if SCALES[col]=="log10" else "HC3 arithmetic mean"
        fig,ax=plt.subplots(figsize=(10.8,6.8))
        xx=np.arange(len(ANALYSIS_LEVELS[a]))
        offsets=np.linspace(-.11,.11,len(ANALYSIS_LEVELS[b]))
        for j,lb in enumerate(ANALYSIS_LEVELS[b]):
            d=grid.loc[grid[bb]==lb]
            xj=xx+offsets[j]
            ax.errorbar(xj,d.estimate,yerr=[d.estimate-d.ci_low,d.ci_high-d.estimate],
                        color=ANALYSIS_COLORS[b][lb],marker="o",ms=7,lw=2.1,capsize=3.5,label=lb)
        ax.set_xticks(xx,ANALYSIS_LEVELS[a]);ax.set_xlabel(FACTOR_LABELS[a]);ax.set_ylabel(f"{estimates_label} of {label} ({unit})")
        if use_log_axis(col): ax.set_yscale("log")
        ax.legend(title=FACTOR_LABELS[b],loc="best",fontsize=9)
        ax.set_title(f"{label}: {FACTOR_LABELS[a]} × {FACTOR_LABELS[b]} interaction",fontsize=16)
        grouping_note = " | grouped BP/BMI categories" if "bp_category" in (a,b) or "bmi_category" in (a,b) else ""
        fig.text(.5,.015,f"HC3 interaction q (42 tests): {model_row.interaction_q_bh_42:.3g} | partial R²: {model_row.interaction_partial_r2:.3g}{grouping_note}",
                 ha="center",fontsize=10)
        fig.tight_layout(rect=(0,.05,1,1))
        save_figure(fig,folder/"03_model_interaction_hc3.png")

        sub=conditional.loc[(conditional.hrv_variable==col)&(conditional.pair==pair)].copy()
        sub=sub.sort_values(["focal_factor","conditioning_factor","conditioning_level","level_1","level_2"])
        if not len(sub): continue
        sub["comparison"]=(sub["level_1"]+" − "+sub["level_2"]+" | "+sub["conditioning_factor"]+"="+sub["conditioning_level"])
        fig,ax=plt.subplots(figsize=(13,max(5.0,.32*len(sub)+1.5)))
        yy=np.arange(len(sub))[::-1]
        for i,(_idx,row) in enumerate(sub.iterrows()):
            color="#b43d58" if row.q_bh_within_hrv_pair<.05 else "#32799f"
            ax.plot([row.cliffs_ci95_low,row.cliffs_ci95_high],[yy[i],yy[i]],color=color,lw=2.0)
            ax.plot(row.cliffs_delta_1_vs_2,yy[i],marker="o",color=color,ms=4.6)
        ax.axvline(0,color="black",lw=.9,ls="--")
        ax.set_yticks(yy,sub["comparison"],fontsize=8)
        ax.set_xlim(-1.03,1.03);ax.set(xlabel="Cliff's delta (95% bootstrap CI)",ylabel="",
                                     title=f"Conditional group effects | {label} | {FACTOR_LABELS[a]} × {FACTOR_LABELS[b]}")
        ax.grid(axis="y",visible=False)
        fig.tight_layout()
        save_figure(fig,folder/"04_conditional_cliffs_forest.png")
print("Saved model interaction and conditional forest figures (84 PNG).")

findfont: Failed to find font weight semibold for DejaVu Sans, now using 700.


Saved model interaction and conditional forest figures (84 PNG).


## 9. Figures — continuous and mixed interactions

For continuous×continuous interactions, the second factor is visualized at its 25th, 50th and 75th percentiles; these percentile slices are **visual reference values**, not additional categorical factors. For interactions with sex, estimates are shown separately for Female and Male. All lines are predictions from the interaction model, with HC3 confidence intervals.

In [10]:
for col,(slug,label,unit) in HRV.items():
    for spec_name,x_raw,z_raw,color_x,color_z in SPECS:
        # Place each continuous specification under its relevant pair.
        factor_of = {"age":"age_group", "sex_category":"sex_category", "imc":"bmi_category",
                     "bp_systolic":"bp_category", "bp_diastolic":"bp_category"}
        canonical_pair = next(pair for pair in PAIRS if set(pair)=={factor_of[x_raw], factor_of[z_raw]})
        folder=FIGURE_DIR/slug/PAIR_NAMES[canonical_pair]
        m=continuous_models[(col,spec_name)]
        result=continuous_results.loc[(continuous_results.hrv_variable==col)&(continuous_results.specification==spec_name)].iloc[0]
        xscaled=CONTINUOUS_DEFS[x_raw][0]
        x_vals=np.linspace(df[x_raw].quantile(.02),df[x_raw].quantile(.98),70)
        fig,ax=plt.subplots(figsize=(10.5,6.7))
        if z_raw=="sex_category":
            zvals=FACTORS["sex_category"]
            labelfmt=lambda z:z
            palette=PALETTES["sex_category"]
            sns.scatterplot(data=df,x=x_raw,y=col,hue=z_raw,hue_order=zvals,palette=palette,alpha=.38,s=21,edgecolor="none",ax=ax,legend=False)
        else:
            qvals=df[z_raw].quantile([.25,.5,.75]).to_numpy()
            zvals=list(qvals)
            unit_z = {"age":"years", "imc":"kg/m²", "bp_systolic":"mmHg", "bp_diastolic":"mmHg"}[z_raw]
            labelfmt=lambda z:f"{CONTINUOUS_DEFS[z_raw][1]} = {z:.1f} {unit_z}"
            palette=dict(zip(zvals,sns.color_palette("crest",len(zvals))))
            # Scatter: color by quantile group of conditioning continuous factor, visual only.
            quantile_labels=["Low ⅓","Middle ⅓","High ⅓"]
            strata=pd.qcut(df[z_raw],q=3,labels=quantile_labels,duplicates="drop")
            hue_pal=dict(zip(quantile_labels,sns.color_palette("crest",3)))
            sns.scatterplot(x=df[x_raw],y=df[col],hue=strata,hue_order=quantile_labels,
                            palette=hue_pal,s=20,alpha=.24,edgecolor="none",legend=False,ax=ax)
        for z in zvals:
            dgrid=pd.DataFrame({xscaled:(x_vals-df[x_raw].mean())/CONTINUOUS_DEFS[x_raw][2]})
            if z_raw=="sex_category":
                dgrid[z_raw]=pd.Categorical([z]*len(dgrid),categories=FACTORS["sex_category"])
            else:
                dgrid[CONTINUOUS_DEFS[z_raw][0]]=(z-df[z_raw].mean())/CONTINUOUS_DEFS[z_raw][2]
            prediction=m.get_prediction(dgrid).summary_frame(alpha=.05)
            est=prediction["mean"].to_numpy();lo=prediction["mean_ci_lower"].to_numpy();hi=prediction["mean_ci_upper"].to_numpy()
            if SCALES[col]=="log10": est,lo,hi=10**est,10**lo,10**hi
            ax.plot(x_vals,est,color=palette[z],lw=2.5,label=labelfmt(z))
            ax.fill_between(x_vals,lo,hi,color=palette[z],alpha=.13)
        if use_log_axis(col):ax.set_yscale("log")
        unit_x = {"age":"years", "imc":"kg/m²", "bp_systolic":"mmHg", "bp_diastolic":"mmHg"}[x_raw]
        z_label = "Sex" if z_raw=="sex_category" else CONTINUOUS_DEFS[z_raw][1]
        ax.set(xlabel=f"{CONTINUOUS_DEFS[x_raw][1]} ({unit_x})",ylabel=f"{label} ({unit})",
               title=f"{label}: {CONTINUOUS_DEFS[x_raw][1]} × {z_label} | HC3 interaction")
        ax.legend(title=z_label,fontsize=9)
        fig.text(.5,.012,f"Interaction q (63 tests): {result.interaction_q_bh_63:.3g} | partial R²: {result.interaction_partial_r2:.3g}",
                 ha="center",fontsize=10)
        fig.tight_layout(rect=(0,.05,1,1))
        save_figure(fig,folder/f"05_continuous_{spec_name}.png",dpi=150)
print(f"Saved continuous/mixed interaction figures: {len(HRV)*len(SPECS)} PNG.")

Saved continuous/mixed interaction figures: 63 PNG.


## 10. Summary heatmaps and quality checks

A heatmap of partial R² indicates the magnitude of the interaction after accounting for both main factors. A separate heatmap indicates HC3 interaction q-values. These complement confidence intervals and group-support tables; stars alone are not taken as evidence of scientific importance.

In [11]:
for col,(slug,label,unit) in HRV.items():
    d=cat_models.loc[cat_models.hrv_variable==col].set_index("pair").reindex([PAIR_NAMES[p] for p in PAIRS])
    c=continuous_results.loc[continuous_results.hrv_variable==col].set_index("specification").reindex([s[0] for s in SPECS])
    fig,ax=plt.subplots(1,2,figsize=(15.8,5.4))
    sns.barplot(x=d.index,y=d.interaction_partial_r2,ax=ax[0],color="#4c879c")
    ax[0].set(title="Categorical interaction partial R²",xlabel="Factor pair",ylabel="Partial R²")
    ax[0].tick_params(axis="x",rotation=40)
    for xx,r in enumerate(d.itertuples()):
        ax[0].text(xx,r.interaction_partial_r2+0.003,f"q={r.interaction_q_bh_42:.2g}",ha="center",fontsize=8)
    sns.barplot(x=c.index,y=c.interaction_partial_r2,ax=ax[1],color="#c78567")
    ax[1].set(title="Continuous/mixed interaction partial R²",xlabel="Specification",ylabel="Partial R²")
    ax[1].tick_params(axis="x",rotation=40)
    fig.suptitle(f"{label} — factorial interaction evidence",fontsize=17,fontweight="semibold")
    fig.tight_layout(rect=(0,0,1,.94),w_pad=2)
    save_figure(fig,FIGURE_DIR/slug/"_summary"/"interaction_evidence.png",dpi=150)

for tab,title,key,value,cmap in [(cat_models,"Categorical interactions","pair","interaction_partial_r2","YlGnBu"),
                                 (continuous_results,"Continuous/mixed interactions","specification","interaction_partial_r2","mako")]:
    v=tab.pivot(index="hrv_variable",columns=key,values=value).reindex(index=list(HRV))
    qcol="interaction_q_bh_42" if key=="pair" else "interaction_q_bh_63"
    q=tab.pivot(index="hrv_variable",columns=key,values=qcol).reindex(index=list(HRV),columns=v.columns)
    ann=np.array([[f"{v.iloc[i,j]:.3f}"+("*" if q.iloc[i,j]<FDR_ALPHA else "")
                   for j in range(v.shape[1])] for i in range(v.shape[0])])
    fig,ax=plt.subplots(figsize=(max(12,1.4*v.shape[1]+4),7.4))
    sns.heatmap(v,cmap=cmap,annot=ann,fmt="",ax=ax,linewidths=.5,
                cbar_kws={"label":"Interaction partial R²"})
    ax.set_yticklabels([HRV[k][1] for k in v.index],rotation=0)
    ax.tick_params(axis="x",rotation=40)
    ax.set(xlabel="Interaction",ylabel="HRV metric",title=f"{title} | partial R² ( * = BH q<0.05 )")
    fig.tight_layout()
    save_figure(fig,GLOBAL_DIR/("categorical_interactions_atlas.png" if key=="pair" else "continuous_interactions_atlas.png"),dpi=165)

# Sanity checks: original data untouched; all six interactions and seven HRV metrics represented.
assert len(df)==476
assert len(PAIRS)==6 and len(HRV)==7
assert len(cat_models)==42 and cat_models.model_status.eq("ok").all()
assert len(continuous_results)==63
assert len(sensitivity)==84
assert support.query("category_scheme == 'analysis'").groupby("pair").n.min().min()>=MIN_MODEL_CELL_N
assert len(list(GLOBAL_DIR.glob("*.png")))==8
assert len(list(FIGURE_DIR.rglob("01_descriptive_overview.png")))==42
assert len(list(FIGURE_DIR.rglob("02_conditioned_ecdf.png")))==42
assert len(list(FIGURE_DIR.rglob("03_model_interaction_hc3.png")))==42
assert len(list(FIGURE_DIR.rglob("04_conditional_cliffs_forest.png")))==42
assert len(list(FIGURE_DIR.rglob("05_continuous_*.png")))==63
assert len(list(FIGURE_DIR.rglob("interaction_evidence.png")))==7
assert len(list(FIGURE_DIR.rglob("*.png"))) == 246
files = sorted(TABLE_DIR.glob("*.csv"))
print("Completed — 07 Bifactorial HRV analysis")
print(f"Input participants: {len(df)}; no changes to input file")
print(f"Categorical models: {len(cat_models)}; continuous/mixed models: {len(continuous_results)}")
print(f"Conditional contrasts: {len(conditional)}; sensitivity refits: {len(sensitivity)}")
print(f"CSV tables: {len(files)}; PNG figures: {len(list(FIGURE_DIR.rglob('*.png')))}")
print(f"Output: {OUTPUT_DIR}")

findfont: Failed to find font weight semibold for DejaVu Sans, now using 700.


Completed — 07 Bifactorial HRV analysis
Input participants: 476; no changes to input file
Categorical models: 42; continuous/mixed models: 63
Conditional contrasts: 511; sensitivity refits: 84
CSV tables: 8; PNG figures: 246
Output: ../../eda_results/07_bifactorial_hrv_analysis
